## Setup

In [ ]:
# Notebook A
!pip install unsloth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 177.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 162.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 52.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 57.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 136.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 125.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 136.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 127.0 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [uns

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

## Step 1 : Data

Take 5,000 rows for training and a separate 200 for testing, and fix the split with a seed so every run uses the same test questions. Each row has three fields: question (English), context (the CREATE TABLE schema), and answer (the correct SQL).

In [ ]:
from datasets import load_dataset

ds = load_dataset("b-mc2/sql-create-context", split="train").shuffle(seed=42)
test_ds  = ds.select(range(200))          # never train on these
train_ds = ds.select(range(200, 5200))
test_ds.to_json("test.jsonl")            # reused in Notebook B
print(train_ds[0])

README.md:   0%|          | 0.00/4.43k [00:00<?, ?B/s]

sql_create_context_v4.json: reconstructing file:   0%|          |  0.00B / 21.8MB            

sql_create_context_v4.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

{'answer': 'SELECT height FROM table_name_69 WHERE name = "dylan postl"', 'question': 'How tall is Dylan Postl?', 'context': 'CREATE TABLE table_name_69 (height VARCHAR, name VARCHAR)'}


Use one prompt format everywhere: training, evaluation, and serving. A mismatch between them is the most common reason a fine-tune "doesn't work".

In [ ]:
SYSTEM = ("You are a SQL expert. Given a table schema and a question, "
          "write one SQLite query that answers it. Return only the SQL.")

def build_messages(ex, with_answer=True):
    msgs = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Schema:\n{ex['context']}\n\nQuestion: {ex['question']}"},
    ]
    if with_answer:
        msgs.append({"role": "assistant", "content": ex["answer"]})
    return msgs

In [ ]:
build_messages(train_ds[0])

[{'role': 'system',
  'content': 'You are a SQL expert. Given a table schema and a question, write one SQLite query that answers it. Return only the SQL.'},
 {'role': 'user',
  'content': 'Schema:\nCREATE TABLE table_name_69 (height VARCHAR, name VARCHAR)\n\nQuestion: How tall is Dylan Postl?'},
 {'role': 'assistant',
  'content': 'SELECT height FROM table_name_69 WHERE name = "dylan postl"'}]

## Step 2


Score the base model on the 200 test questions before any training. This number is what your fine-tune must beat.
Load the model in 4-bit (the same way you will train it):

Load the model in 4-bit (the same way you will train it):

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=1024,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1111 06:49:29.070000 429 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1111 06:49:29.111000 429 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.3: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Generate answers in batches with greedy decoding (do_sample=False), so results are repeatable:

In [ ]:
def generate_sql(model, tokenizer, examples, batch_size=16):
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"
    preds = []
    for i in range(0, len(examples), batch_size):
        batch = examples[i:i + batch_size]
        prompts = [tokenizer.apply_chat_template(build_messages(ex, with_answer=False),
                   tokenize=False, add_generation_prompt=True) for ex in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
        for seq in out[:, inputs["input_ids"].shape[1]:]:
            preds.append(tokenizer.decode(seq, skip_special_tokens=True).strip())
    return preds

test_examples = list(test_ds)   # plain list of dicts

Score with two metrics. Valid SQL runs the query against an empty in-memory database built from the schema. Exact match compares the query to the reference after cleanup.

In [ ]:
import re, sqlite3

def clean(text):
    m = re.search(r"```(?:sql)?\s*(.*?)```", text, re.S)
    return (m.group(1) if m else text).strip()

def normalize(sql):
    return " ".join(sql.strip().rstrip(";").lower().split())

def is_valid(sql, schema):
    con = sqlite3.connect(":memory:")
    try:
        con.executescript(schema)
        con.execute(sql)
        return True
    except Exception:
        return False
    finally:
        con.close()

def score(preds, examples):
    sqls = [clean(p) for p in preds]
    n = len(examples)
    return {
        "exact_match": sum(normalize(s) == normalize(e["answer"]) for s, e in zip(sqls, examples)) / n,
        "valid_sql":   sum(is_valid(s, e["context"]) for s, e in zip(sqls, examples)) / n,
    }

base_preds = generate_sql(model, tokenizer, test_examples)
base_scores = score(base_preds, test_examples)
print(base_scores)

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

{'exact_match': 0.075, 'valid_sql': 0.96}


##Step 3: QLoRA fine-tuning

Train LoRA adapters (rank 16, about 1% of the weights) on top of the frozen 4-bit model for one epoch. On a T4 this usually takes well under an hour for 5,000 short examples.
1. Reload a clean model and add LoRA adapters. Reloading avoids leftovers from the baseline run.

In [ ]:
import torch
del model; torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct", max_seq_length=1024, load_in_4bit=True)

model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
model.print_trainable_parameters()

==((====))==  Unsloth 2026.11.3: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.11.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


2. Turn rows into chat text, plus a small validation set to watch for overfitting:

In [ ]:
val_ds = ds.select(range(5200, 5400))

def to_text(ex):
    return {"text": tokenizer.apply_chat_template(build_messages(ex), tokenize=False)}

train_text = train_ds.map(to_text)
val_text   = val_ds.map(to_text)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

3. Configure and train. Note fp16=True: the T4 has no bfloat16.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,              # newer TRL: processing_class=tokenizer
    train_dataset=train_text,
    eval_dataset=val_text,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=1024,          # newer TRL: max_length
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,   # effective batch = 16
        num_train_epochs=1,
        learning_rate=2e-4,
        warmup_steps=10,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        fp16=True, bf16=False,
        logging_steps=10,
        eval_strategy="steps", eval_steps=50,
        output_dir="outputs",
        report_to="none",             # "wandb" once you set up W&B
        seed=3407,
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/5000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/200 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


4. Train only on the answers. By default the model also learns to repeat the prompt. This Unsloth helper masks everything except the assistant's SQL, which usually improves results:

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

stats = trainer.train()

Map (num_proc=2):   0%|          | 0/5000 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/200 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 313
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.


Step,Training Loss,Validation Loss
50,0.059464,0.062805
100,0.055461,0.057667
150,0.047791,0.054062
200,0.053004,0.043968
250,0.038471,0.041496
300,0.047843,0.040607
313,0.050730,0.040534


Filter:   0%|          | 0/200 [00:00<?, ? examples/s]

Unsloth: Not an error, but `num_items_in_batch` could not be counted for Qwen2ForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-313/tokenizer_config.json.


 save the adapter locally first, before anything else, so a disconnect doesn’t lose your work:

In [ ]:
model.save_pretrained("qwen-sql-lora")
tokenizer.save_pretrained("qwen-sql-lora")


Unsloth: Restored added_tokens_decoder metadata in qwen-sql-lora/tokenizer_config.json.


('qwen-sql-lora/tokenizer_config.json',
 'qwen-sql-lora/chat_template.jinja',
 'qwen-sql-lora/tokenizer.json')

**What to watch**: training loss should drop quickly in the first 50 steps, then flatten. Validation loss should follow it down. If validation loss starts rising while training loss keeps falling, the model is overfitting: train for fewer steps or lower the learning rate.
If you get an out-of-memory error, lower per_device_train_batch_size to 2 and raise gradient_accumulation_steps to 8. The effective batch size stays 16.

##Step 4: Evaluate the fine-tuned model

Run the exact same evaluation code on the same 200 test questions. Only the model changes.

In [ ]:
ft_preds = generate_sql(model, tokenizer, test_examples)
ft_scores = score(ft_preds, test_examples)

print("base:", base_scores)
print("fine-tuned:", ft_scores)

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

base: {'exact_match': 0.075, 'valid_sql': 0.96}
fine-tuned: {'exact_match': 0.72, 'valid_sql': 0.97}


**Read the mistakes**. Numbers say how much; errors say why. Print 10 test questions the fine-tuned model got wrong and sort them into types:

In [ ]:
wrong = [(e, clean(p)) for e, p in zip(test_examples, ft_preds)
         if normalize(clean(p)) != normalize(e["answer"])]
for e, p in wrong[:10]:
    print(e["question"], "\n  expected:", e["answer"], "\n  got:     ", p, "\n")

Who did the Texan's play on Week 4? 
  expected: SELECT opponent FROM table_name_37 WHERE week = "4" 
  got:      SELECT opponent FROM table_name_37 WHERE week = "week 4" 

Which Points have Touchdowns larger than 0, and an Extra points smaller than 0? 
  expected: SELECT SUM(points) FROM table_name_70 WHERE touchdowns > 0 AND extra_points < 0 
  got:      SELECT AVG(points) FROM table_name_70 WHERE touchdowns > 0 AND extra_points < 0 

Of all the contestants who got voted, what is the contestant number and name of the contestant who got least votes? 
  expected: SELECT T1.contestant_number, T1.contestant_name FROM contestants AS T1 JOIN votes AS T2 ON T1.contestant_number = T2.contestant_number GROUP BY T1.contestant_number ORDER BY COUNT(*) LIMIT 1 
  got:      SELECT T1.contestant_number, T2.contestant_name FROM contestants AS T1 JOIN votes AS T2 ON T1.contestant_number = T2.contestant_number ORDER BY T2.votes LIMIT 1 

Find the states where have the colleges whose enrollments are l

##Step 5: Save and publish

Push two versions to the Hub: the small LoRA adapter (about 70 MB) and a merged 16-bit model (about 3 GB). vLLM can serve either, and you will try both in Step 6.

In [ ]:
HF_USER = "harimanikkothu"
token = userdata.get("HF_TOKEN")   # must be a Write token

# 1. Adapter only
model.push_to_hub(f"{HF_USER}/qwen2.5-1.5b-sql-lora", token=token)
tokenizer.push_to_hub(f"{HF_USER}/qwen2.5-1.5b-sql-lora", token=token)

# 2. Adapter merged into the base weights, saved in 16-bit
model.push_to_hub_merged(f"{HF_USER}/qwen2.5-1.5b-sql-merged", tokenizer,
                         save_method="merged_16bit", token=token)

README.md:   0%|          | 0.00/597 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.7kB / 73.9MB            

Validating                    :           |   0%            

Saved model to https://huggingface.co/harimanikkothu/qwen2.5-1.5b-sql-lora


... (16 outputs left out)

  ...-merged/model.safetensors:   1%|          | 22.9MB / 3.09GB            

Validating                    :           |   0%            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [02:21<00:00, 141.42s/it]


Unsloth: Merge process complete. Saved to `/content/harimanikkothu/qwen2.5-1.5b-sql-merged`


In [ ]:
import json
json.dump({"scores": base_scores, "preds": base_preds}, open("baseline.json", "w"))
json.dump({"scores": ft_scores, "preds": ft_preds}, open("finetuned.json", "w"))

from google.colab import files
files.download("baseline.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download("finetuned.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>